In [1]:
import torch

def flash_attention_forward_pass(Q, K, V, br, bc):
    """
    Educational implementation of Flash Attention (Forward Pass).
    
    Args:
        Q, K, V: Tensors of shape (N, d) where N is sequence length, d is dimension.
        br: Block size for rows (Queries).
        bc: Block size for columns (Keys/Values).
    """

    N, d = Q.shape

    # 1. Initialize the output matrix O, the denominator l, and the max m
    # These reside in the "HBM" (main memory)
    O = torch.zeros((N, d))
    l = torch.zeros((N, 1))
    m = torch.full((N, 1), -torch.inf) # Initialize max to negative infinity

    scale = 1.0 / (d ** 0.5)

    # Outer loop: Iterate over blocks of Queries
    for i in range(0, N, br):
        # Load Query block to "SRAM"
        q_i = Q[i:i+br]

        # Load current states for this block to "SRAM"
        m_i = m[i:i+br]
        l_i = l[i:i+br]
        o_i = O[i:i+br]

        # Inner loop: Iterate over blocks of Keys and Values
        for j in range(0, N, bc):
            # Load Key and Value blocks to "SRAM"
            k_j = K[j:j+bc]
            v_j = V[j:j+bc]

            # Step A: Compute unnormalized attention scores for this block
            # Shape: (B_r, B_c)
            s_ij = (q_i @ k_j.T) * scale

            # Step B: Online Softmax Mechanics
            # Find the max value in the current block
            m_ij = torch.max(s_ij, dim=-1, keepdim=True).values

            # Find the new running maximum
            m_new = torch.maximum(m_ij, m_i)

            # Compute the localized exponentials (P)
            p_ij = torch.exp(s_ij - m_new)

            # Step C: The mathematical correction!
            # Because the maximum changed, our previous l_i and o_i are off by an exponential factor.
            # We must correct them before adding the new block's data.
            correction_factor = torch.exp(m_i - m_new)

            # Update the running denominator (l)
            l_new = (correction_factor * l_i) + torch.sum(p_ij, dim=-1, keepdim=True)

            # Update the running unnormalized output (O)
            o_new = (correction_factor * o_i) + (p_ij @ v_j)

            # Save the new states back to our local "SRAM" variables
            m_i = m_new
            o_i = o_new
            l_i = l_new
        
        # End of Inner Loop. 
        # Now that we've looked at all K and V for this Q block, we normalize it.
        # Write the finalized block back to "HBM"
        O[i:i+br] = o_i / l_i
        l[i:i+br] = l_i
        m[i:i+br] = m_i

    return O


In [7]:
import torch.nn.functional as F

torch.manual_seed(42)
N, d = 64, 16  # Keep it small for testing

Q = torch.randn(N, d)
K = torch.randn(N, d)
V = torch.randn(N, d)

# Run our Flash Attention (block sizes of 16)
O_flash = flash_attention_forward_pass(Q, K, V, br=16, bc=16)

# Run standard PyTorch Attention for comparison
scores = (Q @ K.T) / (d ** 0.5)
attention_weights = torch.softmax(scores, dim=-1)
O_standard = attention_weights @ V

# Check if they match mathematically
is_close = torch.allclose(O_flash, O_standard, atol=1e-6)
print(f"Matches standard attention: {is_close}")

lib_attn = F.scaled_dot_product_attention(Q, K, V)

# Check if they match mathematically
is_close2 = torch.allclose(O_flash, lib_attn, atol=1e-6)
print(f"Matches standard attention: {is_close2}")


Matches standard attention: True
Matches standard attention: True
